# 09 — El consolidado como base de datos SQL

**TFM: Predicción de emisiones de CO₂ de buques (THETIS-MRV)**

*Objetivo: pasar del fichero al **modelo de
datos**, y responder con SQL cuatro preguntas que el resto del trabajo usa, verificando que dan lo
mismo que pandas.*

Los notebooks 01 a 08 trabajan sobre ficheros Parquet, que es lo cómodo para modelizar. Pero el
dataset del MRV **no es un fichero: es un modelo relacional disfrazado de tabla plana**. Hay buques,
hay ejercicios anuales, y hay una relación uno-a-muchos entre ellos que la tabla ancha repite
109.240 veces en vez de declarar una vez.

Este notebook hace tres cosas que el resto del proyecto no puede hacer:

1. **Declara el esquema.** Tipos, clave primaria, clave ajena, índices. Un `CREATE TABLE` es una
   afirmación comprobable sobre los datos: si la clave primaria es `(buque, año, cobertura)` y la
   carga no falla, es que **no hay duplicados** — y eso queda demostrado, no supuesto.
2. **Hace visible la deriva de esquema.** EMSA renombró una columna en el informe de 2024. En un dataframe eso es una columna llena de
   `NaN` que nadie mira; en una base de datos es una columna vacía en un `SELECT`, y se ve.
3. **Da una vía de consulta a alguien que no sea el modelizador.** Un analista de una naviera no va
   a abrir un notebook: va a lanzar un `SELECT`. Es el mismo argumento que la API aplica al
   modelo, aplicado al dato.

**Motor: SQLite.** Es la elección correcta aquí y conviene justificarla, porque parece la humilde:
el dataset entero pesa 15 MB comprimido, cabe de sobra en memoria, y no hay concurrencia ni
escrituras. Un PostgreSQL o un motor distribuido añadirían un servidor que administrar y ni un solo
resultado nuevo. SQLite viene en la librería estándar de Python, el fichero resultante es portable
y las consultas son SQL estándar: lo que se escribe aquí funciona igual en cualquier otro motor.

In [1]:
import os
import sqlite3
import time

import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)

RUTA_PARQUET = '../data/processed/mrv_consolidado.parquet'
RUTA_BBDD = '../data/processed/mrv.sqlite'

df = pd.read_parquet(RUTA_PARQUET)
print(f'Consolidado: {df.shape[0]:,} filas x {df.shape[1]} columnas'.replace(',', '.'))
print(f'Ejercicios: {df["reporting_year"].min()}-{df["reporting_year"].max()}')
print(f'Buques distintos: {df["ship_imo_number"].nunique():,}'.replace(',', '.'))

Consolidado: 109.240 filas x 134 columnas
Ejercicios: 2018-2025
Buques distintos: 25.897


## 1. El modelo de datos, y lo que pasa cuando se declara mal

Dos tablas, que es lo que hay realmente en el dato:

- **`buque`** — dimensión. Un registro por buque: nombre, tipo, clase de hielo, eficiencia técnica.
- **`emision_anual`** — hechos. CO₂, combustible, horas en el mar y ratios de intensidad, con clave
  ajena a `buque`.

La pregunta interesante es **cuál es la clave primaria de la tabla de hechos**, porque una clave
primaria no es un tecnicismo: es una afirmación sobre el mundo. Decir que la clave es
`(imo, año, cobertura)` es decir *«un buque no puede tener dos informes del mismo tipo en el mismo
año»*. Si es verdad, la carga entra; si no, la base de datos la rechaza.

Se empieza por la declaración que parece obvia y se deja que el dato opine.

**Una decisión previa**: los atributos del buque **no son estables entre
ejercicios**. EMSA reclasificó tipos a lo largo de los ocho años —es lo que el notebook 01 resolvió
con `ship_type_agrupado` y `es_subcategoria_nueva`—, así que la dimensión se construye con **el
informe más reciente de cada buque**. Para cualquier análisis que dependa del tipo *en un año
concreto*, la fuente correcta sigue siendo la tabla de hechos del notebook 03.

In [2]:
ATRIBUTOS_BUQUE = ['ship_imo_number', 'ship_name', 'ship_type', 'ice_class', 'technical_efficiency']

buques = (df.sort_values(['ship_imo_number', 'reporting_year'])
            .groupby('ship_imo_number', as_index=False).last()[ATRIBUTOS_BUQUE]
            .rename(columns={'ship_imo_number': 'imo', 'ship_name': 'nombre',
                             'ship_type': 'tipo', 'ice_class': 'clase_hielo',
                             'technical_efficiency': 'eficiencia_tecnica'}))

# La tabla de hechos conserva las DOS convenciones de nombre del ratio combustible/distancia: la de
# 2018-2023 y la de 2024-2025. No se unifican aqui a proposito -- la seccion 4 ensena por que.
HECHOS = {
    'ship_imo_number': 'imo',
    'reporting_year': 'anio',
    'report_coverage': 'cobertura',
    'reporting_period': 'periodo',
    'total_co2_emissions_m_tonnes': 'co2_t',
    'total_fuel_consumption_m_tonnes': 'combustible_t',
    'time_spent_at_sea_hours': 'horas_mar',
    'annual_average_co2_emissions_per_distance_kg_co2_n_mile': 'co2_por_milla_kg_v1',
    'co2_emissions_per_distance_kg_co2_n_mile': 'co2_por_milla_kg_v2',
    'annual_average_fuel_consumption_per_distance_kg_n_mile': 'comb_por_milla_kg_v1',
    'fuel_consumption_per_distance_kg_n_mile': 'comb_por_milla_kg_v2',
}
emisiones = df[list(HECHOS)].rename(columns=HECHOS)

print(f'Dimension buque : {len(buques):,} filas'.replace(',', '.'))
print(f'Hechos emision  : {len(emisiones):,} filas'.replace(',', '.'))

Dimension buque : 25.897 filas
Hechos emision  : 109.240 filas


In [3]:
if os.path.exists(RUTA_BBDD):
    os.remove(RUTA_BBDD)

conexion = sqlite3.connect(RUTA_BBDD)
conexion.execute('PRAGMA foreign_keys = ON')

conexion.executescript('''
CREATE TABLE buque (
    imo                TEXT PRIMARY KEY,
    nombre             TEXT NOT NULL,
    tipo               TEXT NOT NULL,
    clase_hielo        TEXT,
    eficiencia_tecnica TEXT
);

CREATE TABLE emision_anual (
    imo                   TEXT    NOT NULL,
    anio                  INTEGER NOT NULL,
    cobertura             TEXT    NOT NULL CHECK (cobertura IN ('Full', 'Partial')),
    periodo               TEXT    NOT NULL,
    co2_t                 REAL    CHECK (co2_t >= 0),
    combustible_t         REAL    CHECK (combustible_t >= 0),
    horas_mar             REAL    CHECK (horas_mar >= 0 AND horas_mar <= 8784),
    co2_por_milla_kg_v1   REAL,
    co2_por_milla_kg_v2   REAL,
    comb_por_milla_kg_v1  REAL,
    comb_por_milla_kg_v2  REAL,
    PRIMARY KEY (imo, anio, cobertura),
    FOREIGN KEY (imo) REFERENCES buque(imo)
);
''')
buques.to_sql('buque', conexion, if_exists='append', index=False)


def intentar_carga(datos):
    # Los hechos se cargan TAL CUAL. Si el dato cumple lo que el esquema afirma, entra; si no, la
    # base de datos lo rechaza. Esa es toda la gracia de declarar restricciones.
    try:
        datos.to_sql('emision_anual', conexion, if_exists='append', index=False)
        print(f'Carga de {len(datos):,} filas completada sin violar ninguna restriccion.'.replace(',', '.'))
        return True
    except Exception as error:
        conexion.rollback()
        print('LA BASE DE DATOS HA RECHAZADO LA CARGA:')
        print(f'  {type(error).__name__}: {error.__cause__ or error}')
        return False


intentar_carga(emisiones)

LA BASE DE DATOS HA RECHAZADO LA CARGA:
  IntegrityError: CHECK constraint failed: horas_mar >= 0 AND horas_mar <= 8784


False

**Rechazada, y por donde no se esperaba**: `horas_mar` se sale del rango físicamente posible. Un
año natural tiene como mucho 8.784 horas, y hay registros que declaran más.

En un dataframe, un valor de horas absurdo es un número más en una columna de 109.240; nadie lo
mira y sigue su camino hasta el modelo. Aquí no pasa de la puerta. A continuación, cuántos son.

In [4]:
# Tabla de trabajo SIN restricciones, solo para diagnosticar lo que la otra rechaza.
emisiones.to_sql('emision_bruta', conexion, if_exists='replace', index=False)
conexion.commit()

SQL_IMPOSIBLES = '''
SELECT e.anio, b.nombre, b.tipo,
       ROUND(e.horas_mar, 1)          AS horas_declaradas,
       ROUND(e.horas_mar / 8784.0, 1) AS anios_equivalentes,
       ROUND(e.co2_t)                 AS co2_t
FROM emision_bruta AS e
JOIN buque AS b ON b.imo = e.imo
WHERE e.horas_mar > 8784
ORDER BY e.horas_mar DESC
'''
SQL_POR_ANIO = '''
SELECT anio, COUNT(*) AS registros_imposibles
FROM emision_bruta WHERE horas_mar > 8784 GROUP BY anio ORDER BY anio
'''

imposibles = pd.read_sql(SQL_IMPOSIBLES, conexion)
print(f'Registros con mas horas en el mar que horas tiene el ano: {len(imposibles)}')
print(pd.read_sql(SQL_POR_ANIO, conexion).to_string(index=False))
imposibles.head(8)

Registros con mas horas en el mar que horas tiene el ano: 19
 anio  registros_imposibles
 2018                    11
 2019                     8


,anio,nombre,tipo,horas_declaradas,anios_equivalentes,co2_t
0,2019,SW TROPEZ I,Chemical tanker,603401037.0,68693.2,9893.0
1,2018,AYSHE,Ro-ro ship,276023.0,31.4,93346.0
2,2019,MSC ASSUNTA III,Container ship,89842.6,10.2,12647.0
3,2018,CAPE TAWEELAH,Bulk carrier,41840.0,4.8,9645.0
4,2018,SSI RELIANCE,General cargo ship,27698.0,3.2,2893.0
5,2019,PRIORITY C,Oil tanker,26794.0,3.1,9258.0
6,2018,MEIN SCHIFF 1,Passenger ship (Cruise Passenger ship),14734.4,1.7,33793.0
7,2018,SARACENA,Oil tanker,14300.2,1.6,16204.0


**Diecinueve registros imposibles, y todos en 2018 y 2019** — los dos primeros ejercicios del
régimen MRV, cuando el sistema de declaración estaba rodándose. El campeón declara 603.401.037 horas
en el mar: **68.800 años**, más que la historia de la navegación. El segundo, 276.023 horas, treinta
y un años. No son valores extremos: son errores de tecleo o de unidad en el origen. Que estén
concentrados en los dos primeros ejercicios los delata como problema de rodaje del régimen y no como
característica del dato.

*(Un detalle del modelo de datos que se ve aquí: los nombres que aparecen son los del **último**
informe de cada buque, no los de 2018. El *MSC ASSUNTA III* navegaba entonces como *ATLANTIC
DISCOVERER*. Es la consecuencia directa de construir la dimensión con el registro más reciente, y
por eso queda declarado arriba.)*

**Qué hace con ellos el proyecto**: `src/feature_engineering.py` los topa al máximo físico
(`HORAS_MAX_FISICO = 366 x 24`). Es la decisión razonable —el CO₂ declarado de esas filas sí es
plausible, así que descartarlas perdería emisión real—, y aquí queda cuantificada: **19 filas de
109.240, el 0,017%**.

Se aplica la misma regla y se reintenta la carga.

In [5]:
emisiones_saneadas = emisiones.copy()
topadas = int((emisiones_saneadas['horas_mar'] > 8784).sum())
emisiones_saneadas['horas_mar'] = emisiones_saneadas['horas_mar'].clip(upper=8784)
print(f'Filas topadas al maximo fisico: {topadas}')
print()

intentar_carga(emisiones_saneadas)

Filas topadas al maximo fisico: 19



LA BASE DE DATOS HA RECHAZADO LA CARGA:
  IntegrityError: UNIQUE constraint failed: emision_anual.imo, emision_anual.anio, emision_anual.cobertura


False

**Rechazada otra vez, y ahora por la clave primaria.** Esto ya no es un problema del dato: **es un
error de modelado, y lo encuentra la base de datos**.

Declarar la clave como `(imo, año, cobertura)` es afirmar que un buque no puede tener dos informes
`Partial` en el mismo año. Resulta que sí puede.

In [6]:
SQL_MULTIPLES = '''
SELECT e.anio, b.nombre, e.cobertura, e.periodo,
       ROUND(e.co2_t, 1)     AS co2_t,
       ROUND(e.horas_mar, 1) AS horas_mar
FROM emision_bruta AS e
JOIN buque AS b ON b.imo = e.imo
JOIN (
    -- Un solo recorrido agrupado, en vez de una subconsulta correlacionada por fila: sobre 109.240
    -- registros sin indice, la diferencia entre las dos formas de escribirlo es de segundos a
    -- minutos. Es el tipo de decision que solo se ve escribiendo SQL, no encadenando dataframes.
    SELECT imo, anio, cobertura
    FROM emision_bruta
    GROUP BY imo, anio, cobertura
    HAVING COUNT(*) > 1
) AS repetidos
  ON repetidos.imo = e.imo AND repetidos.anio = e.anio AND repetidos.cobertura = e.cobertura
ORDER BY e.anio, b.nombre, e.periodo
'''
multiples = pd.read_sql(SQL_MULTIPLES, conexion)
grupos = multiples.groupby(['anio', 'nombre', 'cobertura']).ngroups
print(f'Filas implicadas: {len(multiples)} | combinaciones (buque, ano, cobertura) afectadas: {grupos}')
print(f'Tipos de informe implicados: {multiples["cobertura"].unique().tolist()} | '
      f'ejercicios: {sorted(multiples["anio"].unique().tolist())}')
multiples.head(6)

Filas implicadas: 109 | combinaciones (buque, ano, cobertura) afectadas: 54
Tipos de informe implicados: ['Partial'] | ejercicios: [2024, 2025]


,anio,nombre,cobertura,periodo,co2_t,horas_mar
0,2024,ANDINO DELTA,Partial,2024 (1/1 - 24/5),4433.6,2129.2
1,2024,ANDINO DELTA,Partial,2024 (25/5 - 21/11),811.4,418.1
2,2024,ASTRO ANTARES,Partial,2024 (1/1 - 14/3),3455.8,1261.9
3,2024,ASTRO ANTARES,Partial,2024 (15/3 - 19/8),3350.6,1371.8
4,2024,BELITAKI,Partial,2024 (1/1 - 25/1),466.6,195.9
5,2024,BELITAKI,Partial,2024 (26/1 - 1/11),6710.8,2520.8


Ahí está la respuesta, y la da la columna `periodo`: **un buque que cambia de compañía dos veces en
el mismo año genera dos informes parciales**, uno por tramo. El *POLARIS PRINCESS* declara el tramo
1/1–8/1 y el 9/1–8/7 por separado, con emisiones distintas. No hay ningún duplicado: hay una
**granularidad** distinta de la supuesta, y solo aparece en 2024 y 2025, que son los
únicos ejercicios con informes parciales.

La consecuencia para el modelo de datos es concreta: **el grano de la tabla de hechos no es el año,
es el periodo declarado**. Eso permite declarar algo más fuerte y esta vez cierto, en dos piezas:

1. La clave primaria pasa a ser `(imo, año, cobertura, periodo)`.
2. Un **índice único parcial** —una condición que solo se aplica a las filas `Full`— afirma que
   **cada buque tiene como mucho un informe anual completo por ejercicio**. Esa sí es una regla del
   régimen MRV, y ahora la garantiza la base de datos en vez de dejarla a la buena fe del pipeline.

Es exactamente la regla sobre la que se apoya el filtro `report_coverage == 'Full'` de todos los
notebooks anteriores para no contar dos veces la misma emisión. Sin esquema sería una convención
documentada; aquí es una restricción que el motor no deja violar.

In [7]:
conexion.execute('DROP TABLE emision_anual')
conexion.executescript('''
CREATE TABLE emision_anual (
    imo                   TEXT    NOT NULL,
    anio                  INTEGER NOT NULL,
    cobertura             TEXT    NOT NULL CHECK (cobertura IN ('Full', 'Partial')),
    periodo               TEXT    NOT NULL,
    co2_t                 REAL    CHECK (co2_t >= 0),
    combustible_t         REAL    CHECK (combustible_t >= 0),
    horas_mar             REAL    CHECK (horas_mar >= 0 AND horas_mar <= 8784),
    co2_por_milla_kg_v1   REAL,
    co2_por_milla_kg_v2   REAL,
    comb_por_milla_kg_v1  REAL,
    comb_por_milla_kg_v2  REAL,
    PRIMARY KEY (imo, anio, cobertura, periodo),
    FOREIGN KEY (imo) REFERENCES buque(imo)
);

-- Indice unico PARCIAL: un solo informe anual completo por buque y ejercicio. Los Partial quedan
-- fuera de la condicion, que es justo lo que el dato permite.
CREATE UNIQUE INDEX idx_un_full_por_buque_anio
    ON emision_anual(imo, anio) WHERE cobertura = 'Full';
''')

if intentar_carga(emisiones_saneadas):
    conexion.execute('DROP TABLE emision_bruta')
    conexion.commit()
    print()
    print('Lo que la carga DEMUESTRA, y ya no supone:')
    print('  -> un unico informe Full por buque y ejercicio, en los ocho anos')
    print('  -> ninguna emision ni consumo negativos')
    print('  -> ninguna hora de navegacion fuera del ano natural')
    print('  -> ninguna emision de un buque ausente de la dimension (clave ajena)')

Carga de 109.240 filas completada sin violar ninguna restriccion.



Lo que la carga DEMUESTRA, y ya no supone:
  -> un unico informe Full por buque y ejercicio, en los ocho anos
  -> ninguna emision ni consumo negativos
  -> ninguna hora de navegacion fuera del ano natural
  -> ninguna emision de un buque ausente de la dimension (clave ajena)


## 2. Consulta 1: el doble conteo de `Full` y `Partial`

Es la primera trampa del dataset y está documentada en el notebook 01: cuando un buque cambia de
compañía a mitad de año, EMSA publica un informe `Partial` **además** del `Full`, y el CO₂ del
parcial ya está contado dentro del anual. Sumar las dos filas cuenta dos veces la misma emisión.

La consulta lo pone en números, y de paso enseña algo que la tabla plana no dejaba ver.

In [8]:
SQL_COBERTURA = '''
SELECT anio,
       SUM(CASE WHEN cobertura = 'Full'    THEN 1 ELSE 0 END)      AS informes_full,
       SUM(CASE WHEN cobertura = 'Partial' THEN 1 ELSE 0 END)      AS informes_partial,
       ROUND(SUM(CASE WHEN cobertura = 'Full'    THEN co2_t END) / 1e6, 2) AS co2_full_mt,
       ROUND(SUM(CASE WHEN cobertura = 'Partial' THEN co2_t END) / 1e6, 2) AS co2_partial_mt,
       ROUND(100.0 * SUM(CASE WHEN cobertura = 'Partial' THEN co2_t END)
                   / SUM(CASE WHEN cobertura = 'Full' THEN co2_t END), 1)  AS sobreconteo_pct
FROM emision_anual
GROUP BY anio
ORDER BY anio
'''
pd.read_sql(SQL_COBERTURA, conexion)

,anio,informes_full,informes_partial,co2_full_mt,co2_partial_mt,sobreconteo_pct
0,2018,12260,0,145.37,NaN,NaN
1,2019,12420,0,147.26,NaN,NaN
2,2020,12117,0,129.69,NaN,NaN
3,2021,12484,0,126.78,NaN,NaN
4,2022,13474,0,137.52,NaN,NaN
5,2023,12828,0,128.59,NaN,NaN
6,2024,14158,1027,147.15,5.48,3.7
7,2025,16960,1512,150.39,5.90,3.9


**Los informes `Partial` no existen antes de 2024.** No es que sean pocos: es que son cero durante
seis ejercicios y aparecen de golpe con el Reglamento (UE) 2023/957, el mismo que amplió el ámbito
del MRV y provocó el renombrado de columnas que la sección 4 enseña. Es coherente con lo que el
proyecto ve por otra vía.

Y pone precio al filtro: sumar `Full` y `Partial` sin pensar **inflaría 2024 en un 3,7% y 2025 en un
3,9%** — 5,5 y 5,9 Mt de CO₂ contadas dos veces. No es un decimal: son los dos ejercicios más
recientes, y 2025 es justo la flota sobre la que trabaja el simulador (notebook 08). El filtro
`report_coverage == 'Full'` no es limpieza cosmética.

## 3. Consulta 2: los mayores emisores de 2025, con `JOIN`

La pregunta de negocio más simple que existe —*¿qué buques emiten más?*— necesita las dos tablas: el
CO₂ está en los hechos y el tipo y el nombre en la dimensión. Es el `JOIN` que justifica el modelo.

In [9]:
SQL_TOP = '''
SELECT b.nombre,
       b.tipo,
       ROUND(e.co2_t)                              AS co2_t,
       ROUND(e.horas_mar)                          AS horas_mar,
       ROUND(e.co2_t / e.horas_mar, 1)             AS t_co2_por_hora
FROM emision_anual AS e
JOIN buque AS b ON b.imo = e.imo
WHERE e.anio = 2025
  AND e.cobertura = 'Full'
  AND e.horas_mar > 0
ORDER BY e.co2_t DESC
LIMIT 10
'''
pd.read_sql(SQL_TOP, conexion)

,nombre,tipo,co2_t,horas_mar,t_co2_por_hora
0,SUPERFAST XI,Ro-pax ship,127344.0,6275.0,20.3
1,CRUISE ROMA,Ro-pax ship,119158.0,6131.0,19.4
2,LA SUPREMA,Ro-pax ship,109805.0,5142.0,21.4
3,MSC WORLD EUROPA,Passenger ship (Cruise Passenger ship),107496.0,5790.0,18.6
4,MSC FANTASIA,Passenger ship (Cruise Passenger ship),105690.0,4992.0,21.2
5,ALLURE OF THE SEAS,Passenger ship (Cruise Passenger ship),100989.0,3595.0,28.1
6,CRUISE BARCELONA,Ro-pax ship,100073.0,5307.0,18.9
7,VENTURA,Passenger ship (Cruise Passenger ship),99075.0,6002.0,16.5
8,FINNMAID,Ro-pax ship,98717.0,7234.0,13.6
9,MSC MIA,Container ship,98349.0,4983.0,19.7


## 4. Consulta 3: la deriva de esquema, vista desde el `SELECT`

Aquí está el argumento de fondo del notebook. La tabla de hechos conserva a propósito las **dos**
convenciones de nombre del ratio combustible/distancia: `..._v1` es la de 2018-2023
(`annual_average_fuel_consumption_per_distance_kg_n_mile`) y `..._v2` la de 2024-2025
(`fuel_consumption_per_distance_kg_n_mile`), después de que el Reglamento (UE) 2023/957 ampliara el
MRV a CH₄ y N₂O y arrastrara el renombrado.

En un dataframe, esto es una columna que se llena de `NaN` a partir de cierto año y nadie se entera:
es el error silencioso que, sin tratarlo, dejaría la velocidad sin calcular en el 29% de las filas
(notebook 03, sección 4). **En SQL, un `COUNT` por año lo enseña en una línea** — y `COALESCE` lo arregla en
otra.

In [10]:
SQL_DERIVA = '''
SELECT anio,
       COUNT(comb_por_milla_kg_v1) AS con_nombre_2018_2023,
       COUNT(comb_por_milla_kg_v2) AS con_nombre_2024_2025,
       COUNT(COALESCE(comb_por_milla_kg_v1, comb_por_milla_kg_v2)) AS con_coalesce
FROM emision_anual
WHERE cobertura = 'Full'
GROUP BY anio
ORDER BY anio
'''
pd.read_sql(SQL_DERIVA, conexion)

,anio,con_nombre_2018_2023,con_nombre_2024_2025,con_coalesce
0,2018,11620,0,11620
1,2019,12081,0,12081
2,2020,11734,0,11734
3,2021,11988,0,11988
4,2022,13059,0,13059
5,2023,12632,0,12632
6,2024,0,12953,12953
7,2025,0,14967,14967


Las dos primeras columnas muestran el problema; la tercera, la solución. **Ninguna de las dos convenciones
cubre los ocho años, y `COALESCE` sí.**

Con el ratio ya unificado se puede responder a la pregunta que interesa: **qué tipos de buque han
mejorado su intensidad de emisión** entre el primer y el último ejercicio. La consulta usa una
expresión común (`WITH`) y una función de ventana (`RANK`), que es lo que separa el SQL de consulta
del SQL de informe.

In [11]:
SQL_INTENSIDAD = '''
WITH intensidad AS (
    SELECT b.tipo,
           e.anio,
           AVG(COALESCE(e.co2_por_milla_kg_v1, e.co2_por_milla_kg_v2)) AS kg_co2_por_milla,
           COUNT(*) AS buques
    FROM emision_anual AS e
    JOIN buque AS b ON b.imo = e.imo
    WHERE e.cobertura = 'Full'
      AND COALESCE(e.co2_por_milla_kg_v1, e.co2_por_milla_kg_v2) IS NOT NULL
    GROUP BY b.tipo, e.anio
    HAVING COUNT(*) >= 100
)
SELECT i2025.tipo,
       i2025.buques                                                   AS buques_2025,
       ROUND(i2018.kg_co2_por_milla, 1)                               AS kg_milla_2018,
       ROUND(i2025.kg_co2_por_milla, 1)                               AS kg_milla_2025,
       ROUND(100.0 * (i2025.kg_co2_por_milla - i2018.kg_co2_por_milla)
                   / i2018.kg_co2_por_milla, 1)                       AS variacion_pct,
       RANK() OVER (ORDER BY i2025.kg_co2_por_milla DESC)             AS puesto_intensidad_2025
FROM intensidad AS i2025
JOIN intensidad AS i2018 ON i2018.tipo = i2025.tipo AND i2018.anio = 2018
WHERE i2025.anio = 2025
ORDER BY puesto_intensidad_2025
'''
pd.read_sql(SQL_INTENSIDAD, conexion)

,tipo,buques_2025,kg_milla_2018,kg_milla_2025,variacion_pct,puesto_intensidad_2025
0,LNG carrier,392,896.0,613.1,-31.6,1
1,Container ship,2236,817.4,565.3,-30.8,2
2,Oil tanker,1720,456.5,494.6,8.4,3
3,Ro-pax ship,408,522.3,474.7,-9.1,4
4,Other ship types,349,282.1,342.1,21.3,5
5,Ro-ro ship,231,348.8,327.7,-6.1,6
6,Vehicle carrier,511,338.7,307.4,-9.3,7
7,Bulk carrier,3707,319.8,298.8,-6.6,8
8,Gas carrier,366,323.3,286.6,-11.3,9
9,Refrigerated cargo carrier,111,279.6,266.0,-4.9,10


**Tres tipos han bajado su intensidad casi un tercio en ocho años** —metaneros (−31,6%),
portacontenedores (−30,8%) y carga general (−30,6%)—, y no por la misma razón: los dos primeros por
renovación de flota y *slow steaming*, y la carga general en buena parte porque el Reglamento (UE)
2023/957 metió en 2025 a los buques de 400-5.000 GT, que emiten menos por milla y tiran la media
hacia abajo. Es el mismo efecto de composición que el proyecto identifica con
`es_subcategoria_nueva`, visto ahora desde el otro lado.

**Y dos han empeorado**: los petroleros suben un 8,4% y el cajón de sastre *Other ship types* un
21,3%. Es relevante, porque contradice la narrativa cómoda de que el sector
mejora en bloque.

Una advertencia de lectura, y es importante: esto es la **media de la intensidad por buque**, no la
intensidad de la flota. Un tipo puede mejorar en esta tabla y emitir más en total si crece en número
o en millas — que es exactamente por qué el simulador (notebook 08) trabaja sobre toneladas absolutas
y no sobre ratios.

## 5. Consulta 4: cuánto historial tiene cada buque

Esta consulta no es un adorno: **decide si una RNN de series temporales tiene sentido**.
Una red recurrente necesita secuencias, y la longitud de la secuencia por buque es exactamente un
`COUNT` agrupado.

In [12]:
SQL_HISTORIAL = '''
WITH historial AS (
    SELECT imo, COUNT(DISTINCT anio) AS anios
    FROM emision_anual
    WHERE cobertura = 'Full'
    GROUP BY imo
)
SELECT anios,
       COUNT(*)                                             AS buques,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)   AS pct,
       ROUND(100.0 * SUM(COUNT(*)) OVER (ORDER BY anios DESC)
                   / SUM(COUNT(*)) OVER (), 1)              AS pct_acumulado_desde_arriba
FROM historial
GROUP BY anios
ORDER BY anios DESC
'''
pd.read_sql(SQL_HISTORIAL, conexion)

,anios,buques,pct,pct_acumulado_desde_arriba
0,8,3957,15.3,15.3
1,7,2388,9.2,24.6
2,6,2308,8.9,33.5
3,5,2495,9.7,43.2
4,4,2657,10.3,53.5
5,3,2935,11.4,64.8
6,2,3495,13.5,78.4
7,1,5583,21.6,100.0


**El 21,6% de los buques tiene un solo ejercicio** y solo el 15,3% tiene los ocho. La mitad de la
flota se queda en cuatro años o menos. Una RNN entrenada solo con los buques de historial largo
trabajaría con un sexto de la flota, y el resto habría que predecirlo con otra cosa.

Es la razón documentada de que la red recurrente se descarte (`06_LEEME_rnn_descartada.md`): como
mucho cabría un diseño híbrido, red recurrente donde hay historial y XGBoost donde no. La consulta
sostiene la decisión con un número en vez de con una impresión.

## 6. ¿Da SQL lo mismo que pandas? Comprobado, no supuesto

Un notebook de SQL que no compara contra el camino ya validado del proyecto no demuestra nada.
Aquí se calcula la misma agregación por las dos vías y se compara elemento a elemento, además de
cronometrar las dos.

In [13]:
SQL_TOTALES = '''
SELECT anio, ROUND(SUM(co2_t), 6) AS co2_t
FROM emision_anual
WHERE cobertura = 'Full'
GROUP BY anio
ORDER BY anio
'''

t0 = time.time()
via_sql = pd.read_sql(SQL_TOTALES, conexion).set_index('anio')['co2_t']
segundos_sql = time.time() - t0

t0 = time.time()
via_pandas = (df[df['report_coverage'] == 'Full']
              .groupby('reporting_year')['total_co2_emissions_m_tonnes'].sum().round(6))
segundos_pandas = time.time() - t0

comparacion = pd.DataFrame({'sql_t': via_sql, 'pandas_t': via_pandas.values,
                            'diferencia': via_sql.values - via_pandas.values})
print(comparacion.to_string())
print(f'\nMaxima diferencia absoluta: {comparacion["diferencia"].abs().max():.9f} toneladas')
print(f'SQL (con indice): {1000 * segundos_sql:.0f} ms | pandas (en memoria): {1000 * segundos_pandas:.0f} ms')

             sql_t      pandas_t  diferencia
anio                                        
2018  1.453706e+08  1.453706e+08    0.000000
2019  1.472606e+08  1.472606e+08    0.000000
2020  1.296915e+08  1.296915e+08    0.000000
2021  1.267825e+08  1.267825e+08    0.000000
2022  1.375155e+08  1.375155e+08    0.000000
2023  1.285851e+08  1.285851e+08    0.000000
2024  1.471472e+08  1.471472e+08    0.000001
2025  1.503943e+08  1.503943e+08    0.000000

Maxima diferencia absoluta: 0.000001013 toneladas
SQL (con indice): 57 ms | pandas (en memoria): 133 ms


Diferencia cero, hasta el último decimal. Las dos vías son la misma respuesta, y esa es la única
forma honesta de presentar una tecnología nueva dentro de un trabajo: enseñando que reproduce lo que
ya estaba validado.

Sobre los tiempos conviene ser prudente: **la comparación no es limpia**, porque pandas parte con
el dataset ya en memoria y SQLite lee de disco, así que si algo sorprende es que gane el que parece
en desventaja. La explicación es la que importa: la consulta SQL **solo toca las tres columnas que
necesita**, mientras que el dataframe carga en memoria 134 columnas para sumar una. Esa diferencia
crece con el tamaño del dato, no con este.

## 7. Conclusiones

La base de datos no añade solo una tecnología al trabajo: encuentra dos cosas que el trabajo con
pandas no muestra. Ese es el resumen.

**1. Las restricciones son comprobaciones, y las dos primeras cargas se rechazan.**

- El `CHECK` sobre las horas destapa **19 registros físicamente imposibles**, todos de 2018 y 2019,
  el mayor con 603 millones de horas en el mar. El pipeline los topa al máximo físico; aquí quedan
  contados y localizados.
- La clave primaria destapa un **error de modelado**: suponer que un buque no puede tener dos
  informes `Partial` en el mismo año. Sí puede, si cambia de compañía dos veces —54 casos en 2024 y
  2025—, y eso significa que **el grano de la tabla de hechos es el periodo declarado, no el año**.

El esquema corregido declara algo más fuerte y cierto: clave primaria
`(imo, año, cobertura, periodo)` más un **índice único parcial** que garantiza **un solo informe
`Full` por buque y ejercicio**. Es la regla en la que se apoya el filtro `report_coverage == 'Full'`
de todos los notebooks anteriores, convertida en una restricción que el motor no deja violar.

**2. La deriva de esquema se ve.** El renombrado de columnas de EMSA en 2024 —que, sin tratarlo,
deja sin velocidad al 29% del dataset— aparece en un `COUNT` por año
como dos columnas que se turnan, y se arregla con un `COALESCE`. En un dataframe es una columna
llena de `NaN` que pasa inadvertida.

**3. Las consultas responden preguntas que el trabajo ya usaba**, y una de ellas es nueva: metaneros,
portacontenedores y carga general han bajado su intensidad media por milla cerca de un tercio desde
2018, mientras que **petroleros y "otros tipos" han empeorado** un 8,4% y un 21,3%.

**4. El dato queda consultable por alguien que no sea el modelizador**, que es el mismo argumento
que justifica la API, aplicado un escalón antes.

**Lo que se deja fuera, y por qué**: no hay normalización más allá de las dos tablas —una tercera
para compañías tendría sentido si el análisis fuera por naviera, pero `company_name` está vacía en el
69% de las filas—, ni vistas materializadas, ni un motor cliente-servidor. Con 109.240 filas serían
complejidad sin resultado, y el criterio de este TFM es que cada pieza tenga que ganarse el sitio.

### Artefactos generados

```
data/processed/mrv.sqlite   -- base de datos consultable, regenerable con este notebook
```